In [117]:
# Scan parameters to obtain the optimal ion height and trap frequencies based on RF parameters
## Pothan Tang, 8/6/25
## Compute RF potential taking into account M2 grounding 

import constants as c
import numpy as np
import math
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

## region of interest
um = c.um

## constants
w1 = c.w1
wcs = c.wcs
gap = c.gap
wrf = c.wrf
w2 = c.w2
z_disp = -19*um

## Functions for computing potential
# (x,y,z) = coordinate of sample
# (xi,yi,z0) = ith corner coordinate of electrode
def potential_term(x,y,z,xi,yi,z0):
    num = (xi-x)*(yi-y); # numerator
    den = (z-z0)*math.sqrt((z-z0)**2+(xi-x)**2+(yi-y)**2); # denominator
    return num/den

def dc_potential_single_electrode(x,y,z,x1,y1,x2,y2,z0,v):
    term1 = math.atan(potential_term(x,y,z,x2,y2,z0))
    term2 = math.atan(potential_term(x,y,z,x1,y2,z0))
    term3 = math.atan(potential_term(x,y,z,x2,y1,z0))
    term4 = math.atan(potential_term(x,y,z,x1,y1,z0))
    return (v/(2*math.pi))*(term1-term2-term3+term4)

def dc_potential_total(x,y,z,xy1k,xy2k,z0,vk):
    # extract individual (xik,yik),vk values from array
    # all arrays must be same size
    # sum potentials from all electrodes
    pot = 0
    for i in range (0,len(vk)):
        x1k = xy1k[i][0]
        y1k = xy1k[i][1]
        x2k = xy2k[i][0]
        y2k = xy2k[i][1]
        v = vk[i]
        if(i==19 or i==39): # Q39-40 
            new_pot = dc_potential_single_electrode(x,y,z,x1k,y1k,x2k,y2k,0,v)
            # new_image_pot = dc_potential_single_electrode(x,y,z,x1k,y1k,x2k,y2k,c.z_image_M4,v); # add image potential M2
            # new_pot += new_image_pot
        else:
            new_pot = dc_potential_single_electrode(x,y,z,x1k,y1k,x2k,y2k,z0,v)
        #new_pot = dc_potential_single_electrode(x,y,z,x1k,y1k,x2k,y2k,z0,v)
        pot+=new_pot
    return pot

def compute_rf(lcs_new: float,x: np.ndarray,y: np.ndarray,z: np.ndarray,res: float):
    ## Artificial parameters
    w1_new = 15*um
    gap_new = 0
    wrf_new = 120*um
    vrf_new = 270.85

    ## Computes RF potential for a meshgrid
    phi_rf = np.zeros((len(x),len(y),len(z)),dtype=np.float64); # RF potential initialized to zero
    
    ## Set custom parameters: RF electrode coordinates
    (x11,y11) = (-1*lcs_new/2,-1*(wcs/2+w1_new+gap_new+wrf_new)); # corner 1 (bottom left) of electrode 1
    (x21,y21) = (lcs_new/2,-1*(wcs/2+w1_new+gap_new)); # corner 2 (top right) of electrode 1
    (x12,y12) = (-1*lcs_new/2,wcs/2+w1_new+gap_new); # corner 1 (bottom left) of electrode 2
    (x22,y22) = (lcs_new/2,wcs/2+w1_new+gap_new+wrf_new); # corner 2 (top right) of electrode 2

    ## Compute Potential (V)
    for p in range(0,len(x)):
        xc = x[p]
        for q in range (0,len(y)):
            yc = y[q]
            for r in range (0,len(z)):
                zc = z[r]
                phi_1 = dc_potential_single_electrode(xc,yc,zc,x11,y11,x21,y21,0,vrf_new); # artificial peak voltage 172.6
                phi_2 = dc_potential_single_electrode(xc,yc,zc,x12,y12,x22,y22,0,vrf_new); 
                #phi_1_image = dc_potential_single_electrode(xc,yc,zc,c.x11,c.y11,c.x21,c.y21,c.z_image_M4,c.vrf); # add image potential M2
                #phi_2_image = dc_potential_single_electrode(xc,yc,zc,c.x12,c.y12,c.x22,c.y22,c.z_image_M4,c.vrf); # add image potential M2
                #phi_rf[p][q][r]+=(phi_1_image+phi_2_image)
                phi_rf[p][q][r]+=(phi_1+phi_2)

    ## Compute Pseudopotential (J)
    Ex,Ey,Ez = np.gradient(-1*phi_rf,res,res,res); # Electric field components
    E_squared = Ex**2+Ey**2+Ez**2; # Electric field strength squared
    pseudo = c.e**2/(4*c.m*c.omega**2)*E_squared
    
    return phi_rf,pseudo

def compute_dc(lcs_new: float,x: np.ndarray,y: np.ndarray,z: np.ndarray,res: float):
    ## Computes DC potential for a meshgrid
    phi_dc = np.zeros((len(x),len(y),len(z)), dtype=np.float64); # DC potential initialized to zero

    ## Set custom parameters: DC electrode coordinates
    xy1k_new = c.xy1k
    xy2k_new = c.xy2k
    xy1k_new[19] = (-1*lcs_new/2,wcs/2+w1+gap+wrf+2*gap)
    xy2k_new[19] = (lcs_new/2,wcs/2+w1+gap+wrf+2*gap+w2)
    xy1k_new[39] = (-1*lcs_new/2,-1*(wcs/2+w1+gap+wrf+2*gap+w2))
    xy2k_new[39] = (lcs_new/2,-1*(wcs/2+w1+gap+wrf+2*gap))

    ## compute potential (V)
    for p in range(0,len(x)):
        xc = x[p]
        for q in range (0,len(y)):
            yc = y[q]
            for r in range (0,len(z)):
                zc = z[r]
                phi_dc[p][q][r] = dc_potential_total(xc,yc,zc,xy1k_new,xy2k_new,c.z0,c.vk)

    return phi_dc

In [118]:
# finds local minimum of a 1D array
def find_local_extrema(phi_sliced: np.ndarray):
    delta_phi = [0]
    min_coord = []
    min_val = []
    max_coord = []
    max_val = []
    for i in range(1,len(phi_sliced)):
        delta_phi.append(phi_sliced[i]-phi_sliced[i-1])
        prod = delta_phi[i]*delta_phi[i-1]
        if (prod<0 and delta_phi[i-1]<0):
            min_coord.append(i-1)
            min_val.append(phi_sliced[i-1])
        elif (prod<0 and delta_phi[i-1]>0):
            max_coord.append(i-1)
            max_val.append(phi_sliced[i-1])
    return min_coord,min_val,max_coord,max_val

In [119]:
# Helper function: fit 1D potential to a quadratic
def fit_quadratic(y_vec,y0,energy_vec):
    min_energy = energy_vec[int(len(energy_vec)/2)]
    def quadratic(y_val,A):
        return A*(y_val-y0)**2 + min_energy
    popt, cov = curve_fit(quadratic,y_vec,energy_vec)
    #print(popt)
    # Calculate residuals
    residuals = energy_vec - quadratic(y_vec, *popt)
    ssr = np.sum(residuals**2)
    sst = np.sum((energy_vec - np.mean(energy_vec))**2)
    r_squared = 1 - (ssr / sst)
    #print(f"R-squared: {r_squared:.4f}")
    k = popt[0]*2
    f_trap = np.sqrt(k/c.m)/(2*math.pi)

    return k,f_trap  

Intuition:
1. Ion height is affected by dimensions and placement of RF electrode.
2. Eigenfrequencies are affected by Vrf,omega, and the dimensions and placement of RF electrode.
3. Ion separation is affected by the axial trap strength.
4. Lowering lcs increases fringing, which increases the axial trap frequency.

Initial observations: wrf=110um, lcs=600um, z=-12um gives an ion height of 70um
1. Smaller wrf lowers ion height (can't be too small compared to 110um)
2. Place all electrodes on z=-12um
3. Smaller lcs lowers ion height 
4. Larger lcs ensures that f_trap_y and f_trap_z are approximately equal (lcs around 600um)
5. Decreasing wrf pushes up all trap frequencies

Ideas for tuning parameters:
1. Shorten lcs to lower ion height. Tradeoff: increases axial frequency
2. Move all DC/RF electrodes further down, and vary omega/vrf by a bit to fine tune eigenfrequencies
3. Vary wrf to tune axial trap frequency. Vary vrf to tune radial trap frequencies. Lower all electrodes to tune ion height. Use a smaller mesh to obtain frequencies with higher accuracy.

In [120]:
## Region of interest (wide)
res = 1*um; # resolution of potential 
x_max = 15*um;
y_max = 15*um;
z_max = 200*um;
x = np.linspace(-1*x_max,x_max, int(2*x_max/res+1), endpoint=True); # -1cm<=x<=1cm 
y = np.linspace(-1*y_max,y_max, int(2*y_max/res+1), endpoint=True);  # -1cm<=y<=1cm
z = np.linspace(0,z_max, int(z_max/res+1), endpoint=True);  # 0cm<=z<=1cm

## Scan wrf, gap, lcs, z0
# Radial trap freq decreases with lcs
lcs_original = c.lcs
lcs_vec = np.array([lcs_original])
ion_height = np.zeros(len(lcs_vec))
fx = np.zeros(len(lcs_vec))
fy = np.zeros(len(lcs_vec))
fz = np.zeros(len(lcs_vec))
for i in range(len(lcs_vec)):
    lcs_new = lcs_vec[i]
    # Obtain DC, RF peak and pseudopotential
    phi_rf,pseudo = compute_rf(lcs_new,x,y,z,res)[0],compute_rf(lcs_new,x,y,z,res)[1]
    phi_dc = compute_dc(lcs_new,x,y,z,res)
    energy = c.e*phi_dc+pseudo
    # Compute ion height
    energy_z_axis = energy[15,15,:]
    #print(energy_z_axis)
    z_min_calc = find_local_extrema(1e20*energy_z_axis)[0][0]
    ion_height[i] = z_min_calc
    print(f"For lcs={lcs_new/um}um, calculated minimum trap potential is at z={z_min_calc} um")
    # Compute x-axis axial trap frequency
    pseudo_axial = pseudo[5:26,15,z_min_calc]
    phi_dc_axial = phi_dc[5:26,15,z_min_calc]
    energy_axial = c.e*phi_dc_axial+pseudo_axial
    kx,fx_calc = fit_quadratic(x[5:26],0,energy_axial)
    fx[i] = fx_calc
    print(pseudo_axial)
    print(phi_dc_axial)
    print(energy_axial)
    print(f"For the x-axis, the trap coefficient is {kx} J/m^2 and the trap frequency is {fx_calc} Hz")
    # Compute y-axis radial trap frequency
    pseudo_radial_y = pseudo[15,5:26,z_min_calc]
    phi_dc_radial_y = phi_dc[15,5:26,z_min_calc]
    energy_radial_y = c.e*phi_dc_radial_y+pseudo_radial_y
    ky,fy_calc = fit_quadratic(y[5:26],0,energy_radial_y)
    fy[i] = fy_calc    
    print(pseudo_radial_y)
    print(phi_dc_radial_y)
    print(energy_radial_y)
    print(f"For the y-axis, the trap coefficient is {ky} J/m^2 and the trap frequency is {fy_calc} Hz")
    # Compute z-axis radial trap frequency
    pseudo_radial_z = pseudo[15,15,(z_min_calc-10):(z_min_calc+11)]
    phi_dc_radial_z = phi_dc[15,15,(z_min_calc-10):(z_min_calc+11)]
    energy_radial_z = c.e*phi_dc_radial_z+pseudo_radial_z
    kz,fz_calc = fit_quadratic(z[(z_min_calc-10):(z_min_calc+11)],z_min_calc*um,energy_radial_z)
    fz[i] = fz_calc    
    print(pseudo_radial_z)
    print(phi_dc_radial_z)
    print(energy_radial_z)
    print(f"For the z-axis, the trap coefficient is {kz} J/m^2 and the trap frequency is {fz_calc} Hz")
    
print("Ion height values:")
print(ion_height)
print("x-axis axial trap frequency values:")
print(fx)
print("y-axis radial trap frequency values:")
print(fy)
print("z-axis radial trap frequency values:")
print(fz)

/tmp/ipykernel_17537/3203520573.py:28: RuntimeWarning: divide by zero encountered in scalar divide
  return num/den


For lcs=4139.2um, calculated minimum trap potential is at z=86 um
[1.09186687e-25 1.09172175e-25 1.09159191e-25 1.09147736e-25
 1.09137808e-25 1.09129408e-25 1.09122536e-25 1.09117191e-25
 1.09113373e-25 1.09111082e-25 1.09110319e-25 1.09111082e-25
 1.09113373e-25 1.09117191e-25 1.09122536e-25 1.09129408e-25
 1.09137808e-25 1.09147736e-25 1.09159191e-25 1.09172175e-25
 1.09186687e-25]
[0.1690634  0.16871435 0.16840198 0.16812631 0.16788736 0.16768515
 0.16751969 0.16739099 0.16729905 0.16724389 0.1672255  0.16724389
 0.16729905 0.16739099 0.16751969 0.16768515 0.16788736 0.16812631
 0.16840198 0.16871435 0.1690634 ]
[2.70502537e-20 2.69944049e-20 2.69444254e-20 2.69003185e-20
 2.68620871e-20 2.68297334e-20 2.68032595e-20 2.67826671e-20
 2.67679573e-20 2.67591311e-20 2.67561889e-20 2.67591311e-20
 2.67679573e-20 2.67826671e-20 2.68032595e-20 2.68297334e-20
 2.68620871e-20 2.69003185e-20 2.69444254e-20 2.69944049e-20
 2.70502537e-20]
For the x-axis, the trap coefficient is 5.881960599710

In [121]:
## Region of interest (narrow)- size (31,31,101)
res_small = 0.01*um; # resolution of potential 
x_max_small = 0.15*um;
y_max_small = 0.15*um;
z_max_small = 1*um;
x_small = np.linspace(-1*x_max_small,x_max_small, int(2*x_max_small/res_small+1), endpoint=True); # -1cm<=x<=1cm 
y_small = np.linspace(-1*y_max_small,y_max_small, int(2*y_max_small/res_small+1), endpoint=True);  # -1cm<=y<=1cm
z_small = np.linspace(0+z_min_calc*um,z_max_small+z_min_calc*um, int(z_max_small/res_small+1), endpoint=True);  # 0cm<=z<=1cm

# Obtain DC, RF peak and pseudopotential
phi_rf,pseudo = compute_rf(lcs_original,x_small,y_small,z_small,res_small)[0],compute_rf(lcs_original,x_small,y_small,z_small,res_small)[1]
phi_dc = compute_dc(lcs_original,x_small,y_small,z_small,res_small)
energy = c.e*phi_dc+pseudo
# Compute ion height
energy_z_axis = energy[15,15,:]
print(energy_z_axis)
z_min_calc_accurate = find_local_extrema(1e30*energy_z_axis)[0][0]
ion_height_accurate = z_min_calc+z_min_calc_accurate*0.01
print(f"Calculated minimum trap potential is at z={ion_height_accurate} um")
# Compute x-axis axial trap frequency
pseudo_axial = pseudo[5:26,15,z_min_calc_accurate]
phi_dc_axial = phi_dc[5:26,15,z_min_calc_accurate]
energy_axial = c.e*phi_dc_axial+pseudo_axial
kx,fx_calc = fit_quadratic(x_small[5:26],0,energy_axial)
fx[i] = fx_calc
print(pseudo_axial)
print(phi_dc_axial)
print(energy_axial)
print(f"For the x-axis, the trap coefficient is {kx} J/m^2 and the trap frequency is {fx_calc} Hz")
# Compute y-axis radial trap frequency
pseudo_radial_y = pseudo[15,5:26,z_min_calc_accurate]
phi_dc_radial_y = phi_dc[15,5:26,z_min_calc_accurate]
energy_radial_y = c.e*phi_dc_radial_y+pseudo_radial_y
ky,fy_calc = fit_quadratic(y_small[5:26],0,energy_radial_y)
fy[i] = fy_calc    
print(pseudo_radial_y)
print(phi_dc_radial_y)
print(energy_radial_y)
print(f"For the y-axis, the trap coefficient is {ky} J/m^2 and the trap frequency is {fy_calc} Hz")
# Compute z-axis radial trap frequency
pseudo_radial_z = pseudo[15,15,(z_min_calc_accurate-10):(z_min_calc_accurate+11)]
phi_dc_radial_z = phi_dc[15,15,(z_min_calc_accurate-10):(z_min_calc_accurate+11)]
energy_radial_z = c.e*phi_dc_radial_z+pseudo_radial_z
kz,fz_calc = fit_quadratic(z_small[(z_min_calc_accurate-10):(z_min_calc_accurate+11)],ion_height_accurate*um,energy_radial_z)
fz[i] = fz_calc    
print(pseudo_radial_z)
print(phi_dc_radial_z)
print(energy_radial_z)
print(f"For the z-axis, the trap coefficient is {kz} J/m^2 and the trap frequency is {fz_calc} Hz")

[2.67562485e-20 2.67560545e-20 2.67558991e-20 2.67557540e-20
 2.67556189e-20 2.67554941e-20 2.67553794e-20 2.67552749e-20
 2.67551805e-20 2.67550962e-20 2.67550220e-20 2.67549580e-20
 2.67549040e-20 2.67548601e-20 2.67548263e-20 2.67548026e-20
 2.67547889e-20 2.67547852e-20 2.67547915e-20 2.67548079e-20
 2.67548343e-20 2.67548707e-20 2.67549170e-20 2.67549734e-20
 2.67550397e-20 2.67551159e-20 2.67552021e-20 2.67552982e-20
 2.67554043e-20 2.67555202e-20 2.67556461e-20 2.67557818e-20
 2.67559274e-20 2.67560829e-20 2.67562483e-20 2.67564234e-20
 2.67566085e-20 2.67568033e-20 2.67570080e-20 2.67572224e-20
 2.67574467e-20 2.67576807e-20 2.67579245e-20 2.67581781e-20
 2.67584414e-20 2.67587144e-20 2.67589972e-20 2.67592897e-20
 2.67595919e-20 2.67599038e-20 2.67602254e-20 2.67605566e-20
 2.67608976e-20 2.67612482e-20 2.67616084e-20 2.67619782e-20
 2.67623577e-20 2.67627468e-20 2.67631455e-20 2.67635538e-20
 2.67639717e-20 2.67643991e-20 2.67648361e-20 2.67652827e-20
 2.67657388e-20 2.676620

Parameter Scan: assume all electrodes are at z=-12um, vary lcs for both DC and RF (optimal lcs = 439um)

For lcs=200.0um, calculated minimum trap potential is at z=59 um
For lcs=200.0um, calculated radial trap frequency is 8273049.44485966 Hz
For lcs=300.0um, calculated minimum trap potential is at z=62 um
For lcs=300.0um, calculated radial trap frequency is 15516233.171041781 Hz
For lcs=400.0um, calculated minimum trap potential is at z=64 um
For lcs=400.0um, calculated radial trap frequency is 22034779.24889553 Hz
For lcs=500.00000000000006um, calculated minimum trap potential is at z=66 um
For lcs=500.00000000000006um, calculated radial trap frequency is 27833086.97890402 Hz
For lcs=600.0um, calculated minimum trap potential is at z=68 um
For lcs=600.0um, calculated radial trap frequency is 33041508.092902333 Hz
For lcs=700.0um, calculated minimum trap potential is at z=69 um
For lcs=700.0um, calculated radial trap frequency is 38352756.95814583 Hz
For lcs=2000.0000000000002um, calculated minimum trap potential is at z=71 um
For lcs=2000.0000000000002um, calculated radial trap frequency is 42371495.86360684 Hz
Ion height values:
[59. 62. 64. 66. 68. 69. 71.]
Radial trap frequency values:
[ 8273049.44485966 15516233.17104178 22034779.24889553 27833086.97890402
 33041508.09290233 38352756.95814583 42371495.86360684]

For lcs=435.0um, calculated minimum trap potential is at z=70 um
For lcs=435.0um, calculated radial trap frequency is 3101819.032667552 Hz
For lcs=436.0um, calculated minimum trap potential is at z=70 um
For lcs=436.0um, calculated radial trap frequency is 3100242.5277487477 Hz
For lcs=437.0um, calculated minimum trap potential is at z=70 um
For lcs=437.0um, calculated radial trap frequency is 3098669.936807523 Hz
For lcs=438.0um, calculated minimum trap potential is at z=70 um
For lcs=438.0um, calculated radial trap frequency is 3097087.250257462 Hz
For lcs=439.0um, calculated minimum trap potential is at z=71 um
For lcs=439.0um, calculated radial trap frequency is 3048068.4672011677 Hz
Ion height values:
[70. 70. 70. 70. 71.]
Radial trap frequency values:
[3101819.03266755 3100242.52774875 3098669.93680752 3097087.25025746
 3048068.46720117]

Parameter scan: try omega = 2pi*10^6, z=-12um, vary lcs for both DC and RF (Frequency is too large with small lcs, ion height is too high with large lcs)

For lcs=400.0um, calculated minimum trap potential is at z=69 um
For lcs=400.0um, calculated radial trap frequency is 116836753.09262636 Hz
For lcs=600.0um, calculated minimum trap potential is at z=75 um
For lcs=600.0um, calculated radial trap frequency is 98033906.40914306 Hz
For lcs=800.0um, calculated minimum trap potential is at z=78 um
For lcs=800.0um, calculated radial trap frequency is 89238695.31076351 Hz
For lcs=1000.0000000000001um, calculated minimum trap potential is at z=79 um
For lcs=1000.0000000000001um, calculated radial trap frequency is 85431850.45488325 Hz
For lcs=2000.0000000000002um, calculated minimum trap potential is at z=82 um
For lcs=2000.0000000000002um, calculated radial trap frequency is 77976458.09851664 Hz
Ion height values:
[69. 75. 78. 79. 82.]
Radial trap frequency values:
[1.16836753e+08 9.80339064e+07 8.92386953e+07 8.54318505e+07
 7.79764581e+07]

Parameter scan: test axial trap frequency/eigenfrequencies after shortening both dc and rf electrode length

For lcs=438.0um, calculated minimum trap potential is at z=70 um
For the x-axis, the trap coefficient is 8.257913601313563e-12 J/m^2 and the trap frequency is 858287.3163714202 Hz
For the y-axis, the trap coefficient is 1.075255880009265e-10 J/m^2 and the trap frequency is 3097087.250257462 Hz
For the z-axis, the trap coefficient is 1.0044684917220516e-10 J/m^2 and the trap frequency is 2993406.4245358123 Hz
For lcs=439.0um, calculated minimum trap potential is at z=71 um
For the x-axis, the trap coefficient is 8.18488316589058e-12 J/m^2 and the trap frequency is 854483.6742695974 Hz
For the y-axis, the trap coefficient is 1.0414882669195747e-10 J/m^2 and the trap frequency is 3048068.4672011677 Hz
For the z-axis, the trap coefficient is 9.133078795559281e-11 J/m^2 and the trap frequency is 2854342.799068743 Hz
For lcs=440.0um, calculated minimum trap potential is at z=71 um
For the x-axis, the trap coefficient is 8.17709138335341e-12 J/m^2 and the trap frequency is 854076.8550067263 Hz
For the y-axis, the trap coefficient is 1.0404133641599822e-10 J/m^2 and the trap frequency is 3046495.130697492 Hz
For the z-axis, the trap coefficient is 9.151716967985928e-11 J/m^2 and the trap frequency is 2857253.790011366 Hz
For lcs=600.0um, calculated minimum trap potential is at z=75 um
For the x-axis, the trap coefficient is 7.21679666948532e-12 J/m^2 and the trap frequency is 802360.9094604176 Hz
For the y-axis, the trap coefficient is 8.051089696734209e-11 J/m^2 and the trap frequency is 2679938.6974661537 Hz
For the z-axis, the trap coefficient is 7.768411342293918e-11 J/m^2 and the trap frequency is 2632471.2335277796 Hz
Ion height values:
[70. 71. 71. 75.]
x-axis axial trap frequency values:
[3097087.25025746 3048068.46720117 3046495.13069749 2679938.69746615]
y-axis radial trap frequency values:
[3097087.25025746 3048068.46720117 3046495.13069749 2679938.69746615]
z-axis radial trap frequency values:
[2993406.42453581 2854342.79906874 2857253.79001137 2632471.23352778]

For lcs=439.0um, calculated minimum trap potential is at z=71 um
For the x-axis, the trap coefficient is 8.166944048656648e-12 J/m^2 and the trap frequency is 853546.7585661955 Hz
For the y-axis, the trap coefficient is 1.0413906309709172e-10 J/m^2 and the trap frequency is 3047925.5908765285 Hz
For the z-axis, the trap coefficient is 9.135831036132847e-11 J/m^2 and the trap frequency is 2854772.8427862558 Hz
For lcs=600.0um, calculated minimum trap potential is at z=75 um
For the x-axis, the trap coefficient is 7.21679666948532e-12 J/m^2 and the trap frequency is 802360.9094604176 Hz
For the y-axis, the trap coefficient is 8.051089696734209e-11 J/m^2 and the trap frequency is 2679938.6974661537 Hz
For the z-axis, the trap coefficient is 7.768411342293918e-11 J/m^2 and the trap frequency is 2632471.2335277796 Hz
For lcs=2000.0000000000002um, calculated minimum trap potential is at z=82 um
For the x-axis, the trap coefficient is 6.3335669877303556e-12 J/m^2 and the trap frequency is 751660.4703617468 Hz
For the y-axis, the trap coefficient is 5.0026132855271185e-11 J/m^2 and the trap frequency is 2112496.476926426 Hz
For the z-axis, the trap coefficient is 5.371695029671746e-11 J/m^2 and the trap frequency is 2189037.499703693 Hz
For lcs=4000.0000000000005um, calculated minimum trap potential is at z=83 um
For the x-axis, the trap coefficient is 6.226008411471916e-12 J/m^2 and the trap frequency is 745250.6762905603 Hz
For the y-axis, the trap coefficient is 4.7118847646172015e-11 J/m^2 and the trap frequency is 2050193.5250982814 Hz
For the z-axis, the trap coefficient is 5.0086713659035474e-11 J/m^2 and the trap frequency is 2113775.188734073 Hz
Ion height values:
[71. 75. 82. 83.]
x-axis axial trap frequency values:
[3047925.59087653 2679938.69746615 2112496.47692643 2050193.52509828]
y-axis radial trap frequency values:
[3047925.59087653 2679938.69746615 2112496.47692643 2050193.52509828]
z-axis radial trap frequency values:
[2854772.84278626 2632471.23352778 2189037.49970369 2113775.18873407]

Parameter scan: reduce lcs to tune axial trap frequency, then displace all electrodes vertically -19um to get correct ion height

Before displacement:

For lcs=600.0um, calculated minimum trap potential is at z=87 um
For the x-axis, the trap coefficient is 5.9095580465355654e-12 J/m^2 and the trap frequency is 726064.2085955403 Hz
For the y-axis, the trap coefficient is 8.060358518646606e-11 J/m^2 and the trap frequency is 2681480.8942581723 Hz
For the z-axis, the trap coefficient is 7.889340412349523e-11 J/m^2 and the trap frequency is 2652881.6458947225 Hz
For lcs=625.0um, calculated minimum trap potential is at z=87 um
For the x-axis, the trap coefficient is 5.874906820723049e-12 J/m^2 and the trap frequency is 723932.4075615695 Hz
For the y-axis, the trap coefficient is 7.931897173918424e-11 J/m^2 and the trap frequency is 2660027.1235946976 Hz
For the z-axis, the trap coefficient is 8.068064019874802e-11 J/m^2 and the trap frequency is 2682762.30243062 Hz
For lcs=700.0um, calculated minimum trap potential is at z=88 um
For the x-axis, the trap coefficient is 5.724959950564944e-12 J/m^2 and the trap frequency is 714634.1300082545 Hz
For the y-axis, the trap coefficient is 7.376449506856962e-11 J/m^2 and the trap frequency is 2565199.899534887 Hz
For the z-axis, the trap coefficient is 7.75735714851782e-11 J/m^2 and the trap frequency is 2630597.606781853 Hz
For lcs=800.0um, calculated minimum trap potential is at z=90 um
For the x-axis, the trap coefficient is 5.504639349416591e-12 J/m^2 and the trap frequency is 700748.1541122267 Hz
For the y-axis, the trap coefficient is 6.638662518912804e-11 J/m^2 and the trap frequency is 2433536.267671232 Hz
For the z-axis, the trap coefficient is 6.694057941266444e-11 J/m^2 and the trap frequency is 2443668.3324488755 Hz
Ion height values:
[87. 87. 88. 90.]
x-axis axial trap frequency values:
[2681480.89425817 2660027.1235947  2565199.89953489 2433536.26767123]
y-axis radial trap frequency values:
[2681480.89425817 2660027.1235947  2565199.89953489 2433536.26767123]
z-axis radial trap frequency values:
[2652881.64589472 2682762.30243062 2630597.60678185 2443668.33244888]

After displacement:

For lcs=600.0um, calculated minimum trap potential is at z=68 um
For the x-axis, the trap coefficient is 5.9095580465355654e-12 J/m^2 and the trap frequency is 726064.2085955403 Hz
For the y-axis, the trap coefficient is 8.060358518646606e-11 J/m^2 and the trap frequency is 2681480.8942581723 Hz
For the z-axis, the trap coefficient is 7.889340366148203e-11 J/m^2 and the trap frequency is 2652881.63812686 Hz
For lcs=625.0um, calculated minimum trap potential is at z=68 um
For the x-axis, the trap coefficient is 5.874906820723049e-12 J/m^2 and the trap frequency is 723932.4075615695 Hz
For the y-axis, the trap coefficient is 7.931897173918424e-11 J/m^2 and the trap frequency is 2660027.1235946976 Hz
For the z-axis, the trap coefficient is 8.068064018801068e-11 J/m^2 and the trap frequency is 2682762.302252102 Hz
For lcs=700.0um, calculated minimum trap potential is at z=69 um
For the x-axis, the trap coefficient is 5.724959950564944e-12 J/m^2 and the trap frequency is 714634.1300082545 Hz
For the y-axis, the trap coefficient is 7.376449506856962e-11 J/m^2 and the trap frequency is 2565199.899534887 Hz
For the z-axis, the trap coefficient is 7.757357153078548e-11 J/m^2 and the trap frequency is 2630597.6075551473 Hz
For lcs=800.0um, calculated minimum trap potential is at z=71 um
For the x-axis, the trap coefficient is 5.504639349416591e-12 J/m^2 and the trap frequency is 700748.1541122267 Hz
For the y-axis, the trap coefficient is 6.638662518912804e-11 J/m^2 and the trap frequency is 2433536.267671232 Hz
For the z-axis, the trap coefficient is 6.780104349317124e-11 J/m^2 and the trap frequency is 2459323.8201773665 Hz
Ion height values:
[68. 68. 69. 71.]
x-axis axial trap frequency values:
[2681480.89425817 2660027.1235947  2565199.89953489 2433536.26767123]
y-axis radial trap frequency values:
[2681480.89425817 2660027.1235947  2565199.89953489 2433536.26767123]
z-axis radial trap frequency values:
[2652881.63812686 2682762.3022521  2630597.60755515 2459323.82017737]

After fine tuning radial frequencies:

For lcs=625.0um, calculated minimum trap potential is at z=68 um
For the x-axis, the trap coefficient is 5.900259412021396e-12 J/m^2 and the trap frequency is 725492.7560802788 Hz
For the y-axis, the trap coefficient is 1.076097634542225e-10 J/m^2 and the trap frequency is 3098299.2767461035 Hz
For the z-axis, the trap coefficient is 1.096976466624369e-10 J/m^2 and the trap frequency is 3128212.0359225874 Hz
For lcs=650.0um, calculated minimum trap potential is at z=69 um
For the x-axis, the trap coefficient is 5.787666898106068e-12 J/m^2 and the trap frequency is 718537.2561442583 Hz
For the y-axis, the trap coefficient is 1.0283779693714251e-10 J/m^2 and the trap frequency is 3028823.1038210965 Hz
For the z-axis, the trap coefficient is 1.0188528847009787e-10 J/m^2 and the trap frequency is 3014763.6276377486 Hz
Ion height values:
[68. 69.]
x-axis axial trap frequency values:
[3098299.2767461 3028823.1038211]
y-axis radial trap frequency values:
[3098299.2767461 3028823.1038211]
z-axis radial trap frequency values:
[3128212.03592259 3014763.62763775]